In [ ]:
# Imports
import pandas as pd
import numpy as np
import ast
import xgboost as xgb
import optuna
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error
from collections import Counter

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# Konstanten
file_path = 'data.csv'
columns = [
    # 'id', 'name',
    'host_is_superhost',
    'latitude', 'longitude', 'room_type', 'accommodates',
    'bathrooms', 'bathrooms_text', 'bedrooms', 'beds',
    'amenities', 'price', 'review_scores_rating', 'availability_365', 'neighbourhood_cleansed'
]
# Quelle: https://www.latlong.net/place/london-uk-14153.html
london_center_lat = 51.509865
london_center_lon = -0.118092

#Random Seeds
seeds = [1, 7, 42, 67, 99]

In [ ]:
# CSV einlesen
df = pd.read_csv(file_path)

In [ ]:
pd.set_option('display.max_columns', None)   # alle Spalten zeigen
pd.set_option('display.max_rows', None)      # alle Zeilen zeigen
pd.set_option('display.max_colwidth', None)  # Zellinhalte nicht kürzen
pd.set_option('display.width', None)         # Zeilenbreite automatisch

print(df.columns.tolist())
print(df.head())

In [ ]:
# nur Spalten aus columns werden übernommen + alle Rows mit NaN entfernen
df = df[columns].dropna()

In [ ]:
print(df['neighbourhood_cleansed'].nunique())  # Wie viele einzigartige Werte?
print(df['neighbourhood_cleansed'].value_counts().head(10))

In [ ]:
nb_dummies = pd.get_dummies(df['neighbourhood_cleansed'], prefix='nb', dtype=int)
df = pd.concat([df, nb_dummies], axis=1)
df = df.drop(columns='neighbourhood_cleansed')

In [ ]:
# Vektorisierte Funktion für extrem schnelle Berechnung
def calculate_haversine_distance(lat_series, lon_series, center_lat, center_lon):
    # Erdradius in Metern
    radius_earth = 6371000.0

    # Koordinaten von Grad in Bogenmaß (Radiant) umwandeln
    phi1 = np.radians(lat_series)
    phi2 = np.radians(center_lat)
    delta_phi = np.radians(center_lat - lat_series)
    delta_lambda = np.radians(center_lon - lon_series)

    # Haversine-Formel
    a = np.sin(delta_phi / 2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2.0)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return radius_earth * c

In [ ]:
# Distanz zum Zentrum berechnen
df['distance_to_center_m'] = calculate_haversine_distance(
    df['latitude'],
    df['longitude'],
    london_center_lat,
    london_center_lon
)

# Wert runden
df['distance_to_center_m'] = df['distance_to_center_m'].round()

# Alte Koordinaten-Spalten entfernen
df = df.drop(columns=['latitude', 'longitude'])

In [ ]:
print(df.dtypes)

In [ ]:
# Host_is_superhost in boolean umwandeln
df['host_is_superhost'] = (df['host_is_superhost'] == 't').astype(int)

In [ ]:
# room_type to boolean
print(df.room_type.unique())
df["is_private_room"] = (df["room_type"] == "Private room").astype(int)
df["is_entire_home"]  = (df["room_type"] == "Entire home/apt").astype(int)
df["is_hotel_room"]   = (df["room_type"] == "Hotel room").astype(int)
df["is_shared_room"]  = (df["room_type"] == "Shared room").astype(int)

df = df.drop(columns=['room_type'])

In [ ]:
# 41 hotel rooms und 111 shared rooms droppen
df = df[(df['is_hotel_room'] == False) & (df['is_shared_room'] == False)]

df = df.drop(columns=['is_hotel_room', 'is_shared_room'])

In [ ]:
# Dollarzeichen & andere Sonderzeichen aus "Price" entfernen & in float umwandeln
df['price_dollar'] = (df['price']
                      .str.replace("$", "", regex=False)
                      .str.replace(",", "", regex=False)
                      .str.replace(".00", "", regex=False)
                      .astype(float))
df = df.drop(columns=['price'])

In [ ]:
df['is_shared_bath'] = df['bathrooms_text'].str.contains('shared', case=False, na=False).astype(int)
df = df.drop(columns='bathrooms_text')

In [ ]:
#amenities droppen
df = df.drop(columns='amenities')
# Preise über 1000$ rauswerfen
df = df[df['price_dollar'] < 1000]

In [ ]:
df

In [ ]:
# ── Daten vorbereiten ─────────────────────────────────────────────────────────
X = df.drop(columns='price_dollar')
y = np.log1p(df['price_dollar'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

X_tr, X_val, y_tr, y_val = train_test_split(X_train_scaled, y_train, test_size=0.2, random_state=42)

# ── Optuna Objective ──────────────────────────────────────────────────────────
def objective(trial):
    params = {
        'n_estimators':      1000,
        'early_stopping_rounds': 50,
        'eval_metric':       'rmse',
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
        'random_state':      42,
        'verbosity':         0,
    }

    model = xgb.XGBRegressor(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    preds = model.predict(X_val)
    return root_mean_squared_error(y_val, preds)

# ── Optimierung starten ───────────────────────────────────────────────────────
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)  # erhöhe auf 100 für bessere Ergebnisse

print(f"Bester Val-RMSE: {study.best_value:.4f}")
print(f"Beste Parameter: {study.best_params}")

# ── Finales Modell mit besten Parametern ──────────────────────────────────────
best_params = study.best_params | {
    'n_estimators': 1000,
    'early_stopping_rounds': 50,
    'eval_metric': 'rmse',
    'random_state': 42,
}

final_model = xgb.XGBRegressor(**best_params)
final_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

# ── Test-Evaluation ───────────────────────────────────────────────────────────
y_pred       = np.expm1(final_model.predict(X_test_scaled))
y_test_actual = np.expm1(y_test)

rmse = root_mean_squared_error(y_test_actual, y_pred)
print(f"Test RMSE (in $): {rmse:.2f}")

In [ ]:
importances = pd.Series(final_model.feature_importances_, index=X.columns)
importances.sort_values().tail(20).plot(kind='barh')
plt.tight_layout()
plt.show()

In [ ]:
# Feature-Namen aus dem Booster korrekt mappen
booster = final_model.get_booster()

# Feature-Namen setzen (falls nicht automatisch übernommen)
booster.feature_names = list(X.columns)

# Gain-Importance als Series mit echten Namen
importance_dict = booster.get_score(importance_type='gain')

importances = pd.Series(importance_dict).sort_values()

# Plot
importances.tail(20).plot(kind='barh', figsize=(10, 8))
plt.xlabel('Importance (Gain)')
plt.title('Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
def safe_parse(val):
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return []
    return []

# Häufigkeiten zählen
all_amenities = df["amenities"].apply(safe_parse).explode()
counts = Counter(all_amenities)

# Top 50
top_n = 50
top_amenities = [amenity for amenity, count in counts.most_common(top_n)]
print(top_amenities)

# Nur Top-N im DataFrame behalten
top_set = set(top_amenities)
df["amenities_filtered"] = df["amenities"].apply(safe_parse).apply(
    lambda lst: [a for a in lst if a in top_set]
)